<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/Rank_Variance_Per_Country.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Creating sample data for fb_comments_count
data_comments = {
    'user_id': [1, 2, 1, 2, 3, 3],
    'number_of_comments': [10, 20, 5, 15, 30, 10],
    'created_at': pd.to_datetime(['2019-12-01', '2019-12-15', '2020-01-05', '2020-01-20', '2019-12-10', '2020-01-10'])
}
fb_comments_count = pd.DataFrame(data_comments)

# Creating sample data for fb_active_users
data_users = {
    'user_id': [1, 2, 3],
    'country': ['USA', 'USA', 'UK'],
    'status': ['active', 'active', 'active']
}
fb_active_users = pd.DataFrame(data_users)

display("Comments Table:", fb_comments_count.head())
display("Users Table:", fb_active_users.head())

'Comments Table:'

,user_id,number_of_comments,created_at
0,1,10,2019-12-01
1,2,20,2019-12-15
2,1,5,2020-01-05
3,2,15,2020-01-20
4,3,30,2019-12-10


'Users Table:'

,user_id,country,status
0,1,USA,active
1,2,USA,active
2,3,UK,active


In [2]:
!pip install pandasql
from pandasql import sqldf

# Since pandasql uses SQLite syntax, I've adjusted TO_CHAR to strftime
query = """
WITH cte AS (
    SELECT c.user_id, c.number_of_comments, c.created_at, u.country
    FROM fb_comments_count c
    LEFT JOIN fb_active_users u ON c.user_id = u.user_id
),
de AS (
    SELECT
        strftime('%Y%m', created_at) AS year_month,
        country,
        SUM(number_of_comments) AS total_comments
    FROM cte
    GROUP BY 1, 2
),
final AS (
    SELECT
        year_month, country, total_comments,
        LAG(year_month) OVER (PARTITION BY country ORDER BY year_month) AS prev_year_month,
        LAG(total_comments) OVER (PARTITION BY country ORDER BY year_month) AS prev_total_comments
    FROM de
)
SELECT
    year_month, country, total_comments, prev_year_month, prev_total_comments,
    (total_comments - prev_total_comments) AS comment_difference
FROM final
WHERE prev_year_month='201912' AND year_month='202001' AND (total_comments - prev_total_comments) < 0
ORDER BY country, year_month;
"""

result_df = sqldf(query, locals())
display("Query Result:", result_df)

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=c9e4ef90a7b08d73a3713a8eaff121dd600c74bd6d1fd8dc53e22263332c2021
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


'Query Result:'

,year_month,country,total_comments,prev_year_month,prev_total_comments,comment_difference
0,202001,UK,10,201912,30,-20
1,202001,USA,20,201912,30,-10


### PySpark Implementation
We will now implement the same logic using PySpark. This involves setting up the Spark session and using window functions for the `LAG` operation.

In [3]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize Spark Session
spark = SparkSession.builder.appName('FacebookCommentsAnalysis').getOrCreate()

In [4]:
# 1. Prepare DataFrames from the existing pandas data
spark_comments = spark.createDataFrame(fb_comments_count)
spark_users = spark.createDataFrame(fb_active_users)

# 2. Join and Transform (CTE replacement)
cte_df = spark_comments.join(spark_users, on="user_id", how="left") \
    .withColumn("year_month", F.date_format("created_at", "yyyyMM"))

# 3. Aggregate (de replacement)
de_df = cte_df.groupBy("year_month", "country") \
    .agg(F.sum("number_of_comments").alias("total_comments"))

# 4. Window Functions (final replacement)
window_spec = Window.partitionBy("country").orderBy("year_month")

final_df = de_df.withColumn("prev_year_month", F.lag("year_month").over(window_spec)) \
    .withColumn("prev_total_comments", F.lag("total_comments").over(window_spec))

# 5. Filter and Calculate Difference
result_pyspark = final_df.withColumn("comment_difference", F.col("total_comments") - F.col("prev_total_comments")) \
    .filter((F.col("prev_year_month") == "201912") &
            (F.col("year_month") == "202001") &
            (F.col("comment_difference") < 0)) \
    .orderBy("country", "year_month")

display(result_pyspark.toPandas())

,year_month,country,total_comments,prev_year_month,prev_total_comments,comment_difference
0,202001,UK,10,201912,30,-20
1,202001,USA,20,201912,30,-10


Alternatively, you can run the exact SQL query in Spark by registering the DataFrames as temporary views:

In [6]:
spark_comments.createOrReplaceTempView("fb_comments_count")
spark_users.createOrReplaceTempView("fb_active_users")

# Fix: Spark SQL uses date_format. We need to replace the SQLite strftime syntax.
# Previous attempt failed because 'query' still contained strftime('%Y%m', ...)
spark_sql_query = query.replace("strftime('%Y%m', created_at)", "date_format(created_at, 'yyyyMM')")

try:
    spark_result = spark.sql(spark_sql_query)
    display(spark_result.toPandas())
except Exception as e:
    print(f"Error: {e}")

,year_month,country,total_comments,prev_year_month,prev_total_comments,comment_difference
0,202001,UK,10,201912,30,-20
1,202001,USA,20,201912,30,-10


Solution as per StrataScratch

WITH monthly_comments AS
  (SELECT u.country,
          date_trunc('month', c.created_at)::date AS month_start,
          SUM(c.number_of_comments) AS total_comments
   FROM fb_comments_count AS c
   JOIN fb_active_users AS u ON c.user_id = u.user_id
   WHERE c.created_at >= '2019-12-01'
     AND c.created_at < '2020-02-01'
   GROUP BY u.country,
            date_trunc('month', c.created_at)::date),
     december AS
  (SELECT country,
          total_comments
   FROM monthly_comments
   WHERE month_start = '2019-12-01'),
     january AS
  (SELECT country,
          total_comments
   FROM monthly_comments
   WHERE month_start = '2020-01-01'),
     december_rank AS
  (SELECT country,
          total_comments,
          DENSE_RANK() OVER (
                             ORDER BY total_comments DESC) AS dec_rank
   FROM december),
     january_rank AS
  (SELECT country,
          total_comments,
          DENSE_RANK() OVER (
                             ORDER BY total_comments DESC) AS jan_rank
   FROM january),
     rank_compare AS
  (SELECT d.country,
          d.dec_rank,
          j.jan_rank,
          d.total_comments AS dec_comments,
          j.total_comments AS jan_comments
   FROM december_rank d
   JOIN january_rank j USING (country))
SELECT country
FROM rank_compare
WHERE dec_rank > jan_rank
ORDER BY dec_rank;

Solution build by me
WITH cte AS (
    SELECT
        c.user_id,
        c.number_of_comments,
        c.created_at,
        u.country,
        u.status
    FROM fb_comments_count c
    LEFT JOIN fb_active_users u
        ON c.user_id = u.user_id
),

de AS (
    SELECT
        TO_CHAR(created_at, 'YYYYMM') AS year_month,
        country,
        SUM(number_of_comments) AS total_comments
    FROM cte
    GROUP BY
        TO_CHAR(created_at, 'YYYYMM'),
        country
),

final AS (
    SELECT
        year_month,
        country,
        total_comments,

        LAG(year_month) OVER (
            PARTITION BY country
            ORDER BY year_month
        ) AS prev_year_month,

        LAG(total_comments) OVER (
            PARTITION BY country
            ORDER BY year_month
        ) AS prev_total_comments

    FROM de
)

SELECT
    year_month,
    country,
    total_comments,
    prev_year_month,
    prev_total_comments,
    total_comments - prev_total_comments AS comment_difference
FROM final where prev_year_month='201912' and year_month='202001' and (total_comments - prev_total_comments) <0
ORDER BY
    country,
    year_month;